# **Laboratorio 3**
### ILI-257 PROGRAMACIÓN PARALELA APLICADA
### Profesor: Jorge Díaz M.
### Fecha: 30 de Abril de 2026

Instrucciones: Para esta evaluación debe desarrollar los kernels solicitados en cada una de las preguntas que se presentan a continuación. En cada uno de ellos debe completar en la sección de comentarios explicando que fue lo que hizo junto con el por que se necesitaba dicha solución.

Funciones generales Python utilizadas en esta evaluación.
## ¡No modificar!

In [10]:
from PIL import Image

def png_a_txt(input_png, output_txt):
    img = Image.open(input_png).convert("RGB")  # aseguramos RGB
    width, height = img.size
    pixels = list(img.getdata())  # [(R,G,B), (R,G,B), ...]

    with open(output_txt, "w") as f:
        f.write(f"{width} {height}\n")
        for (r, g, b) in pixels:
            f.write(f"{r} {g} {b} ")

    print(f"Imagen convertida a {output_txt}")

def txt_a_png(input_txt, output_png):
    with open(input_txt, "r") as f:
        width, height = map(int, f.readline().split())
        data = list(map(int, f.read().split()))

    pixels = []
    for i in range(0, len(data), 3):
        pixels.append((data[i], data[i+1], data[i+2]))

    img = Image.new("RGB", (width, height))
    img.putdata(pixels)
    img.save(output_png)

    print(f"Imagen guardada en {output_png}")

## Ejercicio 1 (40pts)



En este primer ejercicio se le pide que, dada una imagen RGB de tamaño potencia de 2 en formato TXT debe contar la cantidad total de pixeles que tiene el color rojo, verde y azul sobre un determinado valor alpha, beta y gamma (un contador por separado para rojo, verde y azul).

Para esto, debe implementar una versión secuencial de esto en CPU (No usando la GPU), y luego dos alternativas para realizar esto en GPU. Posteriormente, debe medir los tiempos de ejecución de cada una de esas alternativas, anotarlos y comentar cual es la mejor alternativa y por que cree que lo es. Además, para las alternativas en GPU, debe detectar cual es el número de hebras por bloque que da el mejor desempeño. Probar tamaños de bloque 16, 32, 64, 128, 256, 512.


Recuerde convertir su imagen "input.png" en TXT ejecutando la siguiente linea

In [11]:
png_a_txt("input.png", "input.txt")

Imagen convertida a input.txt


In [ ]:
%%writefile image_edit.cu
#include <iostream>
#include <fstream>
#include <vector>
#include <cuda_runtime.h>
#include <chrono>

using namespace std;

bool leerTXT(const string& filename, unsigned char*& data, int& width, int& height) {
    ifstream file(filename);
    if (!file) {
        cerr << "Error abriendo archivo\n";
        return false;
    }
    file >> width >> height;
    int size = width * height * 3;
    data = new unsigned char[size];
    for (int i = 0; i < size; i++) {
        int val;
        file >> val;
        data[i] = (unsigned char)val;
    }
    return true;
}

bool escribirTXT(const string& filename, unsigned char* data, int width, int height) {
    ofstream file(filename);
    if (!file) {
        cerr << "Error escribiendo archivo\n";
        return false;
    }
    file << width << " " << height << "\n";
    int size = width * height * 3;
    for (int i = 0; i < size; i++) {
        file << (int)data[i] << " ";
    }
    return true;
}

// Función encargada de contar las casillas marcadas con 255 en el arreglo de entrada:
bool contar(unsigned char* data, int width, int height, const char* etiqueta){
  int size = width * height * 3;
  int r=0, g=0, b=0;
  for (int i = 0; i < size; i++) {
    if ((i % 3) == 0) { // Contar color Rojo
      if (int(data[i]) == 255) {
        r = r + 1;
      }
    }
    else if ((i % 3) == 1) { // Contar color Verde
      if (data[i] == 255) {
        g = g + 1;
      }
    }
    else if ((i % 3) == 2) { // Contar color Azul
      if (data[i] == 255) {
        b = b + 1;
      }
    }
  }
  printf("Colores contabilizados en %s:\n Rojo = %d, Verde = %d, Azul = %d\n", etiqueta, r, g, b);
  return true;
}

// Implementación secuencial en CPU:
bool contar_en_CPU(unsigned char* input, int width, int height, int alpha, int beta, int gamma){
  int size = width * height * 3;
  int r=0, g=0, b=0;
  for (int i = 0; i < size; i++) {
    if ((i % 3) == 0) { // Contar color Rojo
      if (input[i] < alpha) {
        r = r + 1;
      }
    }
    else if ((i % 3) == 1) { // Contar color Verde
      if (input[i] < beta) {
        g = g + 1;
      }
    }
    else if ((i % 3) == 2) { // Contar color Azul
      if (input[i] < gamma) {
        b = b + 1;
      }
    }
  }
  printf("Colores contabilizados en CPU:\n Rojo = %d, Verde = %d, Azul = %d\n", r, g, b);
  return true;
}

// Implementación N°1 en GPU:
__global__ void kernel_GPU1(unsigned char* input, unsigned char* output, int size, int alpha, int beta, int gamma) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < size) {
      for (int k = 0; k < 3; k++) {
        if ((k == 0) && (input[idx*3+k] < alpha)) { // Contar color Rojo
          output[idx*3+k] = 255;
        }
        if ((k == 1) && (input[idx*3+k] < beta)) { // Contar color Verde
          output[idx*3+k] = 255;
        }
        if ((k == 2) && (input[idx*3+k] < gamma)) { // Contar color Azul
          output[idx*3+k] = 255;
        }
      }
    }
}

// Implementación N°2 en GPU:
__global__ void kernel_GPU2(unsigned char* input, unsigned char* output, int size, int alpha, int beta, int gamma) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < size) {
      int canal = idx % 3;

      if ((canal == 0) && (input[idx] < alpha)) { // Contar color Rojo
        output[idx] = 255;
      }
      else if ((canal == 1) && (input[idx] < beta)) { // Contar color Verde
        output[idx] = 255;
      }
      else if ((canal == 2) && (input[idx] < gamma)) { // Contar color Azul
        output[idx] = 255;
      }
    }
}

// Kernel vacío, su finalidad es encender la GPU para obtener una mejor medida de tiempo:
__global__ void kernel_vacio() {
}

int main() {
    cudaError_t err = cudaSuccess;
    string inputFile = "input.txt";
    string outputFileGPU1 = "output1.txt";
    string outputFileGPU2 = "output1_gpu2.txt";

    int width, height;
    unsigned char* h_input = nullptr; ;
    unsigned char* h_output = nullptr;;

    if (!leerTXT(inputFile, h_input, width, height)) {
        return -1;
    }
    int numPixels = width * height;
    int size = numPixels* 3;
    h_output = new unsigned char[size];

    unsigned char *d_input, *d_output;
    err = cudaMalloc((void **)&d_input, size * sizeof(unsigned char) );
	  if (err != cudaSuccess)
	  {
			fprintf(stderr, "Failed to allocate device d_input (error code %s)!\n", cudaGetErrorString(err));
			exit(EXIT_FAILURE);
	  }

    err = cudaMalloc((void **)&d_output, size * sizeof(unsigned char) );
	  if (err != cudaSuccess)
	  {
			fprintf(stderr, "Failed to allocate device d_output (error code %s)!\n", cudaGetErrorString(err));
			exit(EXIT_FAILURE);
	  }

    err = cudaMemcpy(d_input, h_input, size * sizeof(unsigned char), cudaMemcpyHostToDevice);
    if (err != cudaSuccess)
    {
        fprintf(stderr, "Failed to copy d_input from host to device (error code %s)!\n", cudaGetErrorString(err));
        exit(EXIT_FAILURE);
    }

    int threads = 256;
    int blocks = (numPixels + threads - 1) / threads;
    int blocks2 = (size + threads - 1) / threads;

    // Inicialización variables para tomar el tiempo:
    auto start = std::chrono::high_resolution_clock::now();
    auto end = std::chrono::high_resolution_clock::now();
    std::chrono::duration<float, std::milli> ms = end - start;

    // Parámetros modificables, umbrales para contar el color rojo, verde y azul:
    int alpha = 100;
    int beta = 120;
    int gamma = 110;

    // Ejecución 0.0, vacio:
    kernel_vacio<<<1, 1>>>();
    cudaDeviceSynchronize();

    // Ejecución 1.a, secuencial en CPU:
    start = std::chrono::high_resolution_clock::now();
    contar_en_CPU(h_input, width, height, alpha, beta, gamma);
    end = std::chrono::high_resolution_clock::now();
    ms = end - start;
    std::cout << "Tiempo secuencial en CPU: " << ms.count() << " ms\n";

    // Ejecución 1.2, GPU forma 1:
    cudaMemset(d_output, 0, size * sizeof(unsigned char));
    start = std::chrono::high_resolution_clock::now();
    kernel_GPU1<<<blocks, threads>>>(d_input, d_output, numPixels, alpha, beta, gamma);
    cudaDeviceSynchronize();
    end = std::chrono::high_resolution_clock::now();
    ms = end - start;

    err = cudaMemcpy(h_output, d_output, size * sizeof(unsigned char) , cudaMemcpyDeviceToHost);
    if (err != cudaSuccess)
    {
        fprintf(stderr, "Failed to copy h_output from device to host (error code %s)!\n", cudaGetErrorString(err));
        exit(EXIT_FAILURE);
    }

    escribirTXT(outputFileGPU1, h_output, width, height);
    printf("\n");
    contar(h_output, width, height, "GPU, forma N°1");
    std::cout << "Tiempo en GPU forma 1: " << ms.count() << " ms\n";

    // Ejecución 1.3, GPU forma 2:
    cudaMemset(d_output, 0, size * sizeof(unsigned char));
    start = std::chrono::high_resolution_clock::now();
    kernel_GPU2<<<blocks2, threads>>>(d_input, d_output, size, alpha, beta, gamma);
    cudaDeviceSynchronize();
    end = std::chrono::high_resolution_clock::now();
    ms = end - start;

    err = cudaMemcpy(h_output, d_output, size * sizeof(unsigned char) , cudaMemcpyDeviceToHost);
    if (err != cudaSuccess)
    {
        fprintf(stderr, "Failed to copy h_output from device to host (error code %s)!\n", cudaGetErrorString(err));
        exit(EXIT_FAILURE);
    }

    escribirTXT(outputFileGPU2, h_output, width, height);
    printf("\n");
    contar(h_output, width, height, "GPU, forma N°2");
    std::cout << "Tiempo en GPU forma 2: " << ms.count() << " ms\n";

    cudaFree(d_input);
    cudaFree(d_output);
    delete[] h_input;
    delete[] h_output;

    cout << "\nProcesamiento RGB listo -> " << outputFileGPU1 << " y " << outputFileGPU2 << endl;
    return 0;

}

Overwriting image_edit.cu


Recuerde convertir su imagen resultante en PNG ejecutando la siguiente linea

In [14]:
!nvcc -arch=sm_75 image_edit.cu -o image_edit
!./image_edit
print("\n")
txt_a_png("output1.txt", "output1.png")
txt_a_png("output1_gpu2.txt", "output1_gpu2.png")

Colores contabilizados en CPU:
 Rojo = 503512, Verde = 762765, Azul = 736304
Tiempo secuencial en CPU: 19.7445 ms

Colores contabilizados en GPU, forma N°1:
 Rojo = 503512, Verde = 762765, Azul = 736304
Tiempo en GPU forma 1: 0.076025 ms

Colores contabilizados en GPU, forma N°2:
 Rojo = 503512, Verde = 762765, Azul = 736304
Tiempo en GPU forma 2: 0.180493 ms
Procesamiento RGB listo -> output1.txt y output1_gpu2.txt


Imagen guardada en output1.png
Imagen guardada en output1_gpu2.png


## Respuesta
#### Debe explicar lo que hizo y por que lo hizo. Puede extenderse tanto como necesite

- Para hacerlo vía CPU secuencial se

## Ejercicio 2 (60pts)

En este segundo ejercicio se le pide implementar un filtro de procesamiento sobre una imagen RGB de tamaño potencia de 2, entregada en formato TXT.

El filtro a aplicar corresponde a una **convolución 2D (stencil)** sobre la imagen, utilizando una máscara cuadrada de tamaño $(2r + 1) \times (2r + 1)$, donde $r$ es un parámetro entregado al kernel CUDA.

Para cada pixel de salida, el valor de cada canal se calcula de forma independiente como:

$$
O_C(x,y) = \sum_{i=-r}^{r} \sum_{j=-r}^{r} I_C(x+i, y+j) \cdot K(i,j)
$$

donde:
- $C \in \{R, G, B\}$
- $I_C$ representa el canal correspondiente de la imagen de entrada
- $K$ es el kernel de convolución entregado como entrada

Los $K$ que se utilizarán serán dos:
- Promedio simple o suavizado $$K =
\frac{1}{9}
\begin{bmatrix}
1 & 1 & 1 \\
1 & 1 & 1 \\
1 & 1 & 1
\end{bmatrix}
$$
- Edge Detection (o ejemplo lLaplaciano):  $$
K =
\begin{bmatrix}
-1 & -1 & -1 \\
-1 &  8 & -1 \\
-1 & -1 & -1
\end{bmatrix}
$$

Para esto, debe implementar una versión secuencial de esto en CPU (sin ejecución en GPU) y luego implementar las siguientes dos versiones en GPU:

- Versión GPU A:  Cada hebra calcula un pixel de salida (R, G y B) recorriendo toda la ventana de $(2r + 1)² $
- Versión GPU B: La misma definición de convolución que la versión GPU A, pero con una implementación optimizada. Se espera que reduzca el número de accesos a memoria global y evite cálculos redundantes dentro de los bucles internos (como el cálculo repetido de índices).


Ambas versiones deben mantener el mismo modelo de paralelización (1 hebra = 1 pixel de salida), variando únicamente la eficiencia de la implementación en el uso de memoria y cómputo dentro del kernel.


Para ambas versiones GPU probar tamaños de bloque 16, 32, 64, 128, 256, 512. Además, varie el R entre 1 a 3.

Recuerde medir los tiempos de las tres alternativas junto con determinar cuál entrega mejor rendimiento. Explique cómo cree que influye el tamaño del bloque en el rendimiento.

Nota: Para los pixeles fuera del rango de la imagen, asuma que sus valores RGB son 0.


Recuerde convertir su imagen "input.png" en TXT ejecutando la siguiente linea

In [ ]:
png_a_txt("input.png", "input.txt")

Imagen convertida a input.txt


In [ ]:
%%writefile image_edit.cu
#include <iostream>
#include <fstream>
#include <vector>
#include <cuda_runtime.h>

using namespace std;

bool leerTXT(const string& filename, unsigned char*& data, int& width, int& height) {
    ifstream file(filename);
    if (!file) {
        cerr << "Error abriendo archivo\n";
        return false;
    }
    file >> width >> height;
    int size = width * height * 3;
    data = new unsigned char[size];
    for (int i = 0; i < size; i++) {
        int val;
        file >> val;
        data[i] = (unsigned char)val;
    }
    return true;
}

bool escribirTXT(const string& filename, unsigned char* data, int width, int height) {
    ofstream file(filename);
    if (!file) {
        cerr << "Error escribiendo archivo\n";
        return false;
    }
    file << width << " " << height << "\n";
    int size = width * height * 3;
    for (int i = 0; i < size; i++) {
        file << (int)data[i] << " ";
    }
    return true;
}

__global__ void kernel(unsigned char* input, unsigned char* output, int size) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;

    if (idx < size) {

    }
}

int main() {
    cudaError_t err = cudaSuccess;
    string inputFile = "input.txt";
    string outputFile = "output.txt";

    int width, height;
    unsigned char* h_input = nullptr; ;
    unsigned char* h_output = nullptr;;

    if (!leerTXT(inputFile, h_input, width, height)) {
        return -1;
    }
    int numPixels = width * height;
    int size = numPixels* 3;
    h_output = new unsigned char[size];

    unsigned char *d_input, *d_output;
    err = cudaMalloc((void **)&d_input, size * sizeof(unsigned char) );
	  if (err != cudaSuccess)
	  {
			fprintf(stderr, "Failed to allocate device d_input (error code %s)!\n", cudaGetErrorString(err));
			exit(EXIT_FAILURE);
	  }

    err = cudaMalloc((void **)&d_output, size * sizeof(unsigned char) );
	  if (err != cudaSuccess)
	  {
			fprintf(stderr, "Failed to allocate device d_output (error code %s)!\n", cudaGetErrorString(err));
			exit(EXIT_FAILURE);
	  }

    err = cudaMemcpy(d_input, h_input, size * sizeof(unsigned char), cudaMemcpyHostToDevice);
    if (err != cudaSuccess)
    {
        fprintf(stderr, "Failed to copy d_input from host to device (error code %s)!\n", cudaGetErrorString(err));
        exit(EXIT_FAILURE);
    }

    int threads = 256;
    int blocks = (numPixels + threads - 1) / threads;

    kernel<<<blocks, threads>>>(d_input, d_output, numPixels);
    cudaDeviceSynchronize();
    err = cudaMemcpy(h_output, d_output, size * sizeof(unsigned char) , cudaMemcpyDeviceToHost);
    if (err != cudaSuccess)
    {
        fprintf(stderr, "Failed to copy h_output from host to device (error code %s)!\n", cudaGetErrorString(err));
        exit(EXIT_FAILURE);
    }

    escribirTXT(outputFile, h_output, width, height);

    cudaFree(d_input);
    cudaFree(d_output);
    delete[] h_input;
    delete[] h_output;

    cout << "Procesamiento RGB listo -> " << outputFile << endl;
    return 0;

}

Overwriting image_edit.cu


Recuerde convertir su imagen resultante en PNG ejecutando la siguiente linea

In [ ]:
!nvcc -arch=sm_75 image_edit.cu -o image_edit
!./image_edit
txt_a_png("output.txt", "output2.png")

Procesamiento RGB listo -> output.txt
Imagen guardada en output2.png


## Respuesta
#### Debe explicar lo que hizo y por que lo hizo. Puede extenderse tanto como necesite

-